---
**UNIVERSIDAD NACIONAL DE ASUNCIÓN**  
**FACULTAD DE INGENIERÍA**  
# **ROBÓTICA II**
## **Trabajo Práctico Final 2026S1**
# **CaptainP / Captain Pingui R2**
## **Robot cortacésped autónomo**

**Grupo 1:**  
- Judith Giselle Jacquet Boschi  
- Fabrizzio Sebastian Bianchini Morel  
- Victor Gabriel Curiel Gonzalez  
- Ximena Lujan Quenhan Riveros  

**Documento final:** documentación técnica, códigos, videos demo y evidencias de cumplimiento de la guía de TPF.  
**Versión:** 3.0
  
---

## 1. **INTRODUCCIÓN**

**CaptainP** es un prototipo de robot cortacésped autónomo desarrollado como trabajo práctico final de Robótica II. El proyecto busca integrar en una misma plataforma real distintos conceptos de robótica móvil, control, percepción del entorno, comunicación entre sistemas y supervisión remota. Para ello, se diseñó un robot capaz de desplazarse, recibir comandos, monitorear sensores, activar sistemas de corte y utilizar información del entorno para apoyar su navegación.

La arquitectura del sistema se basa en una separación entre control de alto y bajo nivel. Una **Raspberry Pi 4** ejecuta **ROS2** y se encarga de la lógica principal, la comunicación con la aplicación, el procesamiento de información y la coordinación general del robot. Por otro lado, un **ESP32** actúa como controlador de bajo nivel, gestionando motores, sensores, actuadores, señales de seguridad y periféricos físicos.

Además, se desarrolló una **aplicación móvil en Flutter** que permite controlar y supervisar el robot de forma remota, incluyendo modos de operación, visualización de estados y monitoreo de datos relevantes. También se incorporaron módulos de **visión artificial**, orientados a la detección de pasto y obstáculos, con el objetivo de aportar mayor autonomía y capacidad de interpretación del entorno.

En este documento se encuentra el proceso completo de desarrollo de CaptainP: desde la arquitectura general y la selección de componentes, hasta el diseño mecánico, conexionado eléctrico, firmware, nodos ROS2, aplicación móvil, pruebas realizadas, problemas encontrados y posibles mejoras. Aunque el sistema atravesó dificultades propias de una implementación física real, el proyecto permitió validar una plataforma robótica funcional y documentar aprendizajes importantes para futuras mejoras.

## 2. **RESUMEN DEL SISTEMA DESARROLLADO**

**CaptainP** es un robot cortacésped autónomo basado en la siguiente arquitectura:

- Una **Raspberry Pi** ejecuta ROS2 y toma las decisiones de navegación, percepción y comunicación con la app.
- Un **ESP32** ejecuta el firmware de bajo nivel y se encarga de controlar actuadores, leer sensores y aplicar seguridad inmediata.
- Una **app Flutter** permite monitorear el estado del robot, visualizar cámara, enviar comandos y configurar parámetros de operación.
- La **visión artificial** se desarrolló usando Roboflow, YOLOv8 y exportación de modelos para ejecución en Raspberry Pi.

La decisión principal de arquitectura fue separar el sistema en capas:

| Capa | Elemento | Responsabilidad |
|---|---|---|
| Interfaz de usuario | App Flutter | Control, monitoreo, cámara, configuración y geocerca |
| Alto nivel | Raspberry Pi + ROS2 | Navegación, percepción, topics, lógica general y rosbridge |
| Bajo nivel | ESP32 + firmware PlatformIO | Motores, sensores, PID, corte, buzzer, display y emergencia |
| Planta física | Chasis, actuadores y sensores | Desplazamiento, corte, detección física y percepción del entorno |

<br>

Los requisitos solicitados por el profesor de la catedra son:
Matriz de cumplimiento del enunciado

| Requisito de la guía                   | Implementación en CaptainP                    |  
| -------------------------------------- | --------------------------------------------- |
| Tamaño mínimo 40 x 35 cm               | 420 x 480 mm                     |
| Peso mínimo 5 kg                       | aproximadamente 7kg          |
| Dos ruedas actuadas independientemente | Motores DC con encoder + MD13S                |
| Cuchilla de corte                      | Brushless con driver propio                   |
| Trimmer bordeador                      | Motor DC RS555 + MD13S                        |
| 4 bumpers mínimo                       | Finales de carrera: left, right, center, rear |
| GPS OEM                                | NEO-M8                                        |
| 3 ultrasonidos frontales               | Left, center, right                           |
| IMU                                    | MPU6050                                       |
| Autonomía 1 hora                       | Batería actual estimada 	                 |
| Cámara frontal                         | Cámara Raspberry/USB                          |
| Sensor de vuelco                       | IMU por inclinación                           |
| Sensor de levantamiento                | Final de Carrera 			         |
| Botón ON/OFF                           | Interruptor físico                            |
| Buzzer                                 | Buzzer pasivo con eventos                     |

## 3. **OBJETIVOS**

**Objetivo principal**

Desarrollar un robot cortacésped autónomo capaz de navegar, detectar obstáculos y ejecutar tareas de corte de manera segura, utilizando ROS2, un ESP32 para control de bajo nivel y visión artificial para apoyo en la navegación.

**Objetivos específicos**

1. Implementar la arquitectura en capas separando la toma de decisiones (Raspberry Pi), control de actuadores y sensores (ESP32) y la interfaz de usuario (app Flutter).
2. Diseñar una aplicacion que contenga funciones como: control de motores monitoreo en tiempo real del estado de motores, sensores y sistemas de corte, seleccion de modos de corte, etc.
3. Integrar visión artificial mediante modelos de Roboflow y YOLOv8 para la detección de obstáculos y planificación de trayectorias.
4. Configurar la comunicación entre todos los elementos (ROS2, ESP32 y app Flutter) asegurando la transmisión de datos y comandos de manera confiable.

## 4. **ARQUITECTURA DEL ROBOT**

<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=18KnJRe-HVHzNSj9hFyqtJQUFSujTWfTa" width="1000" />
  <p><b>Fig. 1.</b> Diagrama de bloques.</p>
</div>

\
Se decidio utilizar una arquitectura de control jerárquico distribuido, donde la Raspberry Pi actúa como el cerebro del robot, encargándose de la navegación, procesamiento de cámara, gestión de modos y toma de decisiones; mientras que el **ESP32** funciona como firmware de bajo nivel, recibiendo consignas desde la Raspberry y ejecutando el control directo de motores, ESC brushless, sensores, buzzer y periféricos.

Esta arquitectura permite separar el procesamiento de alto nivel del control físico del robot. La Raspberry Pi ejecuta las tareas más pesadas y estratégicas, mientras el ESP32 se encarga de la interacción inmediata con el hardware. De esta forma, el sistema resulta más modular, seguro y fácil de depurar, manteniendo separadas la lógica de navegación y la etapa de potencia.


## 5. **COMPONENTES UTILIZADOS**

La selección de componentes se realizó considerando disponibilidad, costo, facilidad de integración con ROS2/ESP32 y requisitos funcionales del TPF.

### 5.1 Unidad de procesamiento y control

| Componente | Cantidad | Función en el robot | Interfaz principal |
|---|---:|---|---|
| Raspberry Pi 4 | 1 | Computadora principal con ROS2 | USB serial, cámara, red/WiFi |
| ESP32 DevKit | 1 | Controlador de hardware de bajo nivel | UART serial con Raspberry Pi, GPIO, PWM, I2C |
| Cámara Raspberry / USB | 1 | Captura de imágenes para visión artificial | CSI o USB según montaje |
| App Flutter | 1 | Interfaz de usuario | WebSocket hacia rosbridge |

### 5.2 Actuadores

| Actuador | Cantidad | Función | Control |
|---|---:|---|---|
| Motores DC con encoder | 2 | Tracción diferencial del robot | Driver de potencia + PID en ESP32 |
| Motor DC RS555 | 1 | Trimmer / corte inferior | PWM desde ESP32 mediante driver |
| Motor BLDC Nema 23 | 1 | Cuchilla principal | Señal PWM/VR al driver brushless |
| Buzzer pasivo | 1 | Alertas sonoras | PWM / tonos desde ESP32 |
| Display OLED SSD1309/SSD1306 | 1 | Estado local del robot | I2C |

### 5.3 Sensores

| Sensor | Cantidad | Función | Interfaz |
|---|---:|---|---|
| Ultrasonidos HC-SR04 | 3 | Detección de obstáculos frontal/lateral | GPIO TRIG/ECHO |
| Finales de carrera | 4 | Bumpers / colisión física | GPIO con `INPUT_PULLUP` |
| IMU MPU6050 | 1 | Aceleración y velocidad angular | I2C |
| GPS NEO-M8 | 1 | Posición global y velocidad | UART2 |

### 5.4 Alimentación

| Elemento | Cantidad | Uso |
|---|---:|---|
| Batería Li-Ion 12 V | 1 | Fuente principal seleccionada |
| Reguladores DC-DC | Según necesidad | Conversión a tensiones lógicas y auxiliares |

\
ENLACE DE PRUEBA DE LOS COMPONENTES CON ROS2: https://youtube.com/playlist?list=PLBBiOttPCWPs&si=YTynOw2D-B0L0Byw


## 6. **DISEÑO**

### 6.1 Criterio de diseño general

A la hora de hacer el diseño se tuvo en cuenta unos cuantos criterios como:
1. Facilidad de ensamblaje.
2. Optimizacion de espacio.
3. Evitar el ruido electrico debido a vibraciones y componentes que podrian producirlo.
4. Funcionalidad.
5. Requisitos de diseño del profesor (peso, elementos, etc.)
6. Estetica.

En la [Fig. 2] se puede observar el primer diseño del robot, que luego de analisis el ruido electrico y facilidad de ensamblaje y conexion sufrio de diversos cambios en cuanto a distribucion principalmente.

En la [Fig. 3] tenemos el diseño actual del robot, el que se monto para la realizacion de pruebas de programacion. Lo unico que esta faltando seria el diseño del mecanismo que activaria los finales de carrera, lo cual seria el bumper y la tapa del robot que seria principalmente estetico y para proteccion del sistema electrico.

En las primeras etapas se evaluó una estructura metálica, pero se optó por una solución con base de madera/prototipo por tres motivos principales:

1. Permite modificar agujeros, soportes y distribución de componentes con rapidez.
2. Reduce problemas de interferencia eléctrica y contacto accidental.
3. Facilita el montaje de módulos durante la etapa de prueba.


### 6.2 Distribución funcional del montaje

| Zona del robot | Elementos ubicados | Justificación |
|---|---|---|
| Parte inferior / central | Motores de tracción, ruedas y soporte estructural | Mantener bajo el centro de masa y transmitir fuerza al suelo |
| Zona frontal | Ultrasonidos, bumpers y cámara | Detección anticipada de obstáculos |
| Zona central superior | Raspberry Pi, ESP32 y drivers | Acceso para conexión, programación y pruebas |
| Zona de corte | Trimmer y cuchilla principal | Separación física respecto a la electrónica |
| Zona de batería | Pack Li-Ion y soporte | Balance de peso y acceso para carga/desmontaje |

### 6.3 Evidencias visuales del diseño y montaje

Las siguientes imágenes provienen de las entregas parciales y sirven como evidencia del proceso de diseño, montaje y validación.


<table>
  <tr>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1_Xt2J6mdGc3uWujchv9pTbJLpDczrjng" width="500">
      <br>
      <b>Fig. 2.</b> Ensamblaje del diseño completo V1.0.
    </td>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1jMgL1FCQnTpPVIvPorwnlcCtFfpf1N5K" width="500">
      <br>
      <b>Fig. 3.</b> Ensamblaje del diseño completo V2.0 (final)
    </td>
  </tr>
</table>

\
Las siguientes imagenes muestran como el robot, tras las mejoras realizadas luego de diversas pruebas, quedo armado:

<table>
  <tr>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1ahQBvHxwfli6JZpbigc_CL_pSG00eyVB" width="400">
      <br>
      <b>Fig. 4.</b> Montaje Final: Vista Superior.
    </td>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1kt9ElrNjv3jnBr8OZszLmVxB7a6kTxnZ" width="400">
      <br>
      <b>Fig. 5.</b> Montaje Final: Vista Frontal.
    </td>
  </tr>
</table>

<table>
  <tr>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1t5JcgxDtrXHgl4x4KJjEVsaLCTG-Jq4z" width="400">
      <br>
      <b>Fig. 6.</b> Montaje Final: Vista Inferior.
    </td>
    <td align="center" width="50%">
      <img src="https://drive.google.com/uc?export=view&id=1d70bYZRnqS88MyZq3iBN0nHE1bNMEKB9" width="400">
      <br>
      <b>Fig. 7.</b> Montaje Final: Vista Isometrica.
    </td>
  </tr>
</table>


## 7. **CONEXIONADO ELECTRICO**

Para las conexiones electricas se utilizaron cables de 20AWG que soportan hasta 3A aproximadamente, por seguridad se opto por agregar fusibles entre cada conexion importante de potencia por ejemplo entre el switch de alimentacion y la conexion general, entre la alimentacion y driver del motor, etc.

Adi tambien tomamos la decision de apantallar los cables de señal y alimentacion de los componentes mas sensibles a EMI e incluso separar las tierras de potencia y logica.

**Diagrama de conexion**

En el diagrama se pueden observar todos los componentes colocados segun la ubicacion que ocupan en el robot. Los caminos naranjas simulan ser los cables de alimentacion de 24V, los lilas son de 5V, el verde es 3.3V y el marron es GND.

La finalidad del diagrama es ayudar a repartir mejor el cableado dentro del robot para evitar enredos y complicaciones a la hora de verificar conexionado.


<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=191yIkQ2QGjJewK_FM5XAXHf3DdLQGzus" width="1000" />
  <p><b>Fig. 8.</b> Diagrama de conexiado realizado en drawio.</p>
</div>

**Pin Out ESP32 y del Raspberry*

Con ayuda de la IA generamos una imagen del pinout del ESP32 y del Raspberry utilizado con etiquetas de las señales a conectar, cada una colocada en su respectivo GPIO y por colores para mejor distinción.

<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=185bohEevU6MjfJwEY6jg-U4TanXjZI_-" width="500" />
  <p><b>Fig. 9.</b> PinOut ESP32.</p>
</div>


<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=1YGUhR2sW223CDS7z8qyLrVtf1_IvT1Rv" width="500" />
  <p><b>Fig. 10.</b> PinOut Raspberry.</p>
</div>

## 8. **VISION ARTIFICIAL**

La visión artificial del robot quedó organizada en dos partes principales: la detección de pasto y la detección de obstáculos. Para esto se utilizó una cámara con stream MJPEG, la cual es tomada por ROS2 y luego enviada a los nodos encargados del procesamiento.

La arquitectura final utilizada fue la siguiente:

```text
Cámara MJPEG
      |
      v
camera_bridge_node
      |
      v
/camera/image_raw
      |
      +------------------------+
      |                        |
      v                        v
grass_detector_node     object_detection_node
      |                        |
      v                        v
/grass_state            /vision_obstacle_state
/grass/debug_image      /obstacle/debug_image
      |                        |
      v                        v
blade_node              App
      |
      v
serial_node
      |
      v
   ESP32

```

### 8.1 Herramientas utilizadas

| Herramienta | Uso |
|---|---|
| Roboflow | Organización y descarga de datasets |
| YOLO | Entrenamiento y pruebas iniciales |
| Modelos .pt | Formato intermedio antes de exportar |
| TensorFlow Lite | Exportación para usar los modelos en la Raspberry Pi |
| OpenCV | Procesamiento de imágenes y dibujo de resultados |
| ROS2 | Comunicación entre nodos |
| Raspberry Pi | Ejecución de visión artificial |
| ESP32 | Control físico de actuadores |

Para la parte de visión se trabajó con Roboflow para organizar los datasets. Se usaron modelos diferentes para la clasificación de pasto y para la detección de objetos. Primero se trabajaron los modelos en formato `.pt` y después se exportaron a `.tflite`.

Se utilizó el formato TensorFlow Lite float16 porque era más liviano y se adaptaba mejor a la Raspberry Pi. Esto ayudó a que el procesamiento sea más rápido y no consuma tantos recursos.


### 8.2 Cámara y publicación de imágenes

El nodo `camera_bridge_node` se encarga de leer el stream MJPEG de la cámara. Luego convierte cada imagen a un mensaje de ROS2 y lo publica en el tópico:

`/camera/image_raw`

Este tópico es usado por los demás nodos de visión, así no hace falta abrir la cámara varias veces desde distintos procesos.

### 8.3 Detección de pasto

La detección de pasto se realiza con el nodo `grass_detector_node`. Este nodo recibe la imagen desde `/camera/image_raw` y analiza una región de interés en la parte inferior de la imagen, que corresponde a la zona más cercana al robot.

El modelo utilizado fue:

`best_float16_grass.tflite`

Las clases usadas fueron:

- pasto
- no pasto

El resultado se publica en el tópico:

`/grass_state`

El formato del mensaje es:

`etiqueta,confianza`

Por ejemplo:

`pasto,0.873`

También se publica una imagen de depuración en:

`/grass/debug_image`

Esta imagen permite ver en la app qué parte de la imagen se está analizando y cuál fue el resultado obtenido.

<div align="center">
  <img src="https://drive.google.com/uc?id=1cPFq-hvbYNTZpvyk2sEs1X01nn5Svc-Q" width="600" />
  <p><b>Fig. 11.</b> Visualización de detección de pasto en la aplicación.</p>
</div>

<div align="center">
  <img src="https://drive.google.com/uc?id=1lxLGT2xhSmBooCRrjmuKkvW6sgixboni" width="600" />
  <p><b>Fig. 12.</b> Visualización de detección de ausencia de pasto en la aplicación.</p>
</div>



### 8.4 Detección de obstáculos

La detección de obstáculos se realiza con el nodo `object_detection_node`. Este nodo recibe la imagen desde `/camera/image_raw` y analiza una sola región de la imagen. Dentro de esa región el modelo detecta los objetos usando bounding boxes.

El modelo utilizado fue:

`best_float16_obstacle.tflite`

La clase principal trabajada fue:

- obstaculo

El estado de la detección se publica en:

`/vision_obstacle_state`

Además, el nodo publica una imagen de depuración en:

`/obstacle/debug_image`

En esta imagen se puede ver el bounding box sobre el obstáculo detectado y la confianza del modelo. Esta salida es la que se muestra en la aplicación para verificar visualmente si el sistema está detectando correctamente.

La detección de obstáculos quedó funcionando para visualización en la app. La idea inicial era usarla también para navegación con campo potencial, pero no se llegó a integrar en esta etapa.



<div align="center">
  <img src="https://drive.google.com/uc?id=1yHdHZe8MzHmg8t27TxCZYMh3dSRhWLnL" width="600" />
  <p><b>Fig. 13.</b> Prueba de detección de obstáculo desde la cámara.</p>
</div>

### 8.5 Integración con el corte

El nodo `blade_node` es el encargado de controlar el brushless de corte usando la información que llega desde visión, navegación y la aplicación.

Este nodo se suscribe a los siguientes tópicos:

`/grass_state`

`/vision/enabled`

`/navegacion/corte`

`/vision/manual_blade_enable`

El tópico `/grass_state` viene desde `grass_detector_node` y contiene el resultado de la detección de pasto. El formato usado es:

`etiqueta,confianza`

Por ejemplo:

`pasto,0.873`

El tópico `/vision/enabled` indica si la visión está activada o desactivada desde la aplicación. Cuando la visión está activada, el corte depende de si se detecta pasto o no.

El tópico `/navegacion/corte` indica si la navegación está solicitando activar el sistema de corte. También se agregó el tópico `/vision/manual_blade_enable`, que permite activar el brushless manualmente desde la aplicación.

La lógica usada en el nodo fue:

- Si no hay solicitud de corte, el brushless queda apagado.
- Si se activa el corte manual desde la app, el brushless se prende.
- Si la navegación solicita corte y la visión está desactivada, el brushless se prende.
- Si la navegación solicita corte y la visión está activada, el brushless solo se prende cuando se detecta `pasto`.
- Si se detecta `no pasto`, `none` o no llegan datos recientes de visión, el brushless se apaga por seguridad.

El comando final hacia el ESP32 se publica en:

`/esp32/cmd_blade`

Cuando el valor publicado es `True`, el nodo de comunicación serial envía al ESP32:

`CMD_BRUSHLESS,1`

Cuando el valor publicado es `False`, se envía:

`CMD_BRUSHLESS,0`

También se hizo que el nodo solo publique un nuevo comando cuando cambia el estado deseado. Esto evita enviar muchas veces el mismo comando por serial sin necesidad.

De esta forma, la Raspberry Pi toma la decisión usando ROS2 y visión artificial, mientras que el ESP32 se encarga de activar o apagar físicamente el brushless.

### 8.6 Visualización en la aplicación

La aplicación muestra las imágenes de depuración generadas por los nodos de visión:

`/grass/debug_image`

`/obstacle/debug_image`

En la imagen de pasto se puede ver la región que analiza el modelo y el resultado obtenido. En la imagen de obstáculos se muestra el bounding box sobre el objeto detectado, junto con la confianza del modelo.

También desde la aplicación se puede activar o desactivar el uso de visión para el corte. Ese estado llega al sistema como:

`/vision/enabled`

Además, se agregó una opción para habilitar el brushless manualmente desde la app mediante:

`/vision/manual_blade_enable`

Esto permite probar el sistema de corte de forma manual o usando la detección de pasto, según lo que se necesite durante las pruebas.

### 8.7 Prueba de funcionamiento con visión

En la siguiente prueba se puede ver el funcionamiento del sistema de visión conectado con el control del brushless. Cuando la cámara detecta `pasto`, el nodo `blade_node` activa el brushless. Cuando la cámara detecta `no pasto`, el brushless se apaga.

<div align="center">
  <video width="600" controls>
    <source src="https://drive.google.com/uc?id=1bP4wW6Uq1lqqx0lyRDSjx0GMaQM_GkDJ" type="video/mp4">
  </video>
  <p><b>Fig. 14.</b> Prueba de activación del brushless mediante visión artificial.</p>
</div>

Esta prueba muestra que la salida del nodo `grass_detector_node` se usa correctamente para controlar el corte. El resultado publicado en `/grass_state` llega al nodo `blade_node`, y desde ahí se envía el comando correspondiente al ESP32 para prender o apagar el brushless.

## 9. **FIRMWARE (ESP32)**

## 9.1 Rol del ESP32

El ESP32 es la capa de bajo nivel del robot: no toma decisiones de navegación,
sino que ejecuta las órdenes que le llegan por serial desde ROS2 (Raspberry Pi)
y devuelve telemetría. Se encarga del control de tracción con PID y encoders,
de los motores de corte, de los sensores (ultrasonidos, IMU, batería), del
buzzer, del display y de la parada de emergencia.

## 9.2 Estructura del firmware

El proyecto está armado en PlatformIO con Arduino Framework, con una clase por
periférico:

```
FIRMWARE_CAPTAINP/
├── include/          (un .h por módulo + pinmap.h)
├── src/
│   ├── main.cpp             instancias globales, setup() y loop()
│   ├── DriveBase.cpp        tracción: PID + encoders
│   ├── TrimmerController.cpp
│   ├── BrushlessController.cpp
│   ├── CutSequencer.cpp     secuencia de arranque de los motores de corte
│   ├── UltrasonicManager.cpp
│   ├── ImuManager.cpp
│   ├── BatteryManager.cpp
│   ├── BuzzerController.cpp
│   ├── DisplayController.cpp
│   ├── SafetyManager.cpp    bandera de emergencia
│   └── SerialProtocol.cpp   parser de comandos
└── platformio.ini
```

Todos los pines y constantes están centralizados en `pinmap.h`: ningún módulo
define pines propios, lo que facilita recablear sin tocar la lógica.

## 9.3 Protocolo serial ESP32 - Raspberry Pi

Serial USB a 115200 baud. Los comandos son líneas de texto y cada uno se
responde con `ACK` o `ERR`.

| Comando | Función |
|---|---|
| `PING` | Prueba de conexión |
| `STATUS` | Solicita la telemetría completa |
| `CMD_MOTORES,left_rpm,right_rpm` | Setpoint de RPM para tracción diferencial |
| `CMD_TRIMMER,0/1` | Enciende/apaga trimmer |
| `CMD_BRUSHLESS,0/1` | Enciende/apaga brushless |
| `CMD_DISPLAY,text` | Muestra el modo de operación en el OLED |
| `CMD_BUZZER,text` | Ejecuta un patrón de sonido |
| `CMD_ESTOP` | Activa la emergencia |
| `CMD_RESET_ESTOP` | Libera la emergencia |
| `IMU_CALIBRATE` | Calibra la IMU (robot quieto y plano) |
| `IMU_RESET_YAW` | Pone el yaw en cero |
| `RESET_ENCODERS` | Reinicia los ticks de encoders |

| Telemetría | Datos |
|---|---|
| `ULTRASONICOS,left,center,right,diag_right,diag_left,back_right` | Distancias en cm (-1 = sin lectura) |
| `MOTORES,time_ms,left_rpm,right_rpm,left_ticks,right_ticks` | Estado de la tracción |
| `TRIMMER,state` / `BRUSHLESS,state` | Estado de los motores de corte |
| `IMU,ax,ay,az,gx,gy,gz` | IMU cruda |
| `IMU_ANGLES,roll,pitch,yaw,calibrated` | Orientación calculada |
| `BATTERY,volts` | Tensión de batería |
| `BUZZER,text` / `DISPLAY,text` | Último estado de buzzer y display |
| `ESTADO,estop_state` | Estado de emergencia |

## 9.4 Funcionamiento interno

El `loop()` no usa ningún `delay()`: cada módulo tiene un `update()` que avanza
por tiempo o por interrupciones, así el ciclo principal gira libre y la lectura
serial nunca se atrasa.

**Tracción.** Cada rueda tiene un PID de velocidad propio. Los encoders se leen
por interrupciones en cuadratura (3168 cuentas por vuelta de rueda) y la RPM
medida se suaviza con un filtro exponencial antes de entrar al PID. La salida
va como PWM de 20 kHz a los drivers MD13S, con compensación de fricción
estática y anti-windup en la integral. Como failsafe, si no llega un
`CMD_MOTORES` nuevo en 500 ms la tracción se frena sola, y durante los primeros
2 segundos tras el encendido los motores quedan bloqueados.

**Ultrasonidos.** Los 6 HC-SR04 comparten el pin TRIG y el eco de cada uno se
mide por interrupciones, sin bloquear el loop. Se muestrea un sensor por turno
cada 40 ms y un filtro descarta los ecos fantasma: un salto brusco de distancia
solo se acepta si se repite en la lectura siguiente.

**IMU.** El MPU6050 se lee por I2C y la orientación (roll, pitch, yaw) se
calcula con un filtro complementario: giroscopio a corto plazo, acelerómetro a
largo plazo. Cuando la vibración de los motores de corte aleja la magnitud del
acelerómetro de 1g, esa corrección se descarta. La calibración se guarda en la
flash del ESP32, así que sobrevive a los apagados.

**Batería.** Se acumula una muestra de ADC por ciclo; cada 500 muestras se
promedia y el resultado pasa por una mediana y un suavizado exponencial para
eliminar el ruido eléctrico.

**Motores de corte.** Trimmer y brushless arrancan y frenan con rampas suaves,
y un secuenciador escalona los arranques (ruedas → 1 s → brushless → 2 s →
trimmer) para no exigirle picos de corriente a la batería.

## 9.5 Parada de emergencia

Los finales de carrera los lee un nodo de ROS2 en la Raspberry Pi. Ante un
contacto, envía `CMD_ESTOP`: el firmware apaga de inmediato tracción y motores
de corte, y rechaza todo comando de movimiento con `ERR ESTOP_ACTIVE` hasta
recibir `CMD_RESET_ESTOP`. Así la decisión de frenar puede venir tanto de los
sensores como de la lógica de navegación, pero la ejecución del frenado es
siempre local e inmediata en el ESP32.



## 10. **SOFTWARE ALTO NIVEL (Raspberry Pi 4)**

### 10.1 Principio de integración

La Raspberry Pi ejecuta ROS2 y se comunica con el ESP32 por serial. Para evitar conflictos, **solo un nodo debe abrir el puerto serial del ESP32**:

```text
pkg_esp32_bridge/esp32_bridge_node.py
```

Los demás nodos no acceden al puerto serial. En su lugar, publican o escuchan topics ROS2.

```text
Nodos ROS2 → topics → esp32_bridge_node → Serial USB → ESP32
ESP32 → Serial USB → esp32_bridge_node → topics → Nodos ROS2
```

### 10.2 Workspace del proyecto

Para este proyecto se utiliza el workspace:

```bash
~/brain_captainp
```


### 10.3 Paquetes y nodos principales

<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=1PcFviStXKfHtgfVjVor_Ro17dt77mXGQ" width="500" />
  <p><b>Fig. 15.</b> Gráfico RQT de los nodos y tópicos del ROS2.</p>
</div>

El gráfico generado con el comando rqt_graph nos permite visualizar la arquitectura de comunicación del sistema ROS2 del robot. En este diagrama se observan los nodos activos, los tópicos que publican y los tópicos a los que se suscriben, representando con flechas el flujo de información entre ellos. Por ejemplo, se puede ver cómo el esp32_bridge_node funciona como intermediario entre el ESP32 y ROS2, publicando datos de sensores como ultrasonidos, IMU, batería y estados de actuadores, y recibiendo comandos para motores, trimmer, cuchilla y buzzer. También se observa la conexión de los nodos de navegación, como random_node, perimetral_node y parallel_lines_node, con los tópicos de sensores, cerco virtual, modo de la app y comandos hacia los motores. Además, el gráfico muestra la integración con la aplicación mediante rosbridge_websocket, la cámara y los nodos de visión artificial. En conjunto, rqt_graph fue útil para comprobar que los nodos estaban correctamente conectados y que la información fluía entre sensores, procesamiento, navegación, actuadores y aplicación móvil.

Para comprender de mejor manera realizamos una explicación más detallada de cada paquetes y sus respectivos nodos y tópicos.

#### `pkg_esp32_bridge`

| Nodo | Función |
|---|---|
| `esp32_bridge_node.py` | Único nodo que abre serial, traduce comandos ROS2 a serial y telemetría serial a topics |

Como en el ESP32 se concentra la conexión directa con los sensores y actuadores del robot y actua como controlador de bajo nivel. Desde ahí se leen constantemente los datos de ultrasonidos, encoders, RPM de motores, IMU, batería y estados de actuadores. Toda esa información agrupada se envia por comunicación serial hacia la Raspberry Pi. Luego, en la Raspberry, el nodo `esp32_bridge_node` recibe esos mensajes seriales, los interpretaba y separaba la información para publicarla en los tópicos correspondientes de ROS2, como `/sensors/ultrasonic`, `/sensors/limit_switches`, `/motors/rpm`, `/imu/angles` o `/battery/voltage`. De forma inversa, cuando algún nodo de ROS2 necesita accionar motores, trimmer, cuchilla, buzzer, display o comandos especiales, publicaba en tópicos como `/esp32/cmd_motores`, `/esp32/cmd_trimmer` o `/esp32/cmd_status`, y el bridge convierte esas órdenes en comandos seriales para enviarlos nuevamente al ESP32.


- Publicaciones principales:

| Topic | Tipo | Origen |
|---|---|---|
| `/esp32/serial_rx` | `String` | `ESP32` |
| `/esp32/serial_tx` | `String` | `ESP32` |
| `/esp32/ack` | `String` | `ESP32` |
| `/esp32/error` | `String` | `ESP32` |
| `/sensors/ultrasonic` | `Float32MultiArray` | `ULTRASONICOS` |
| `/sensors/ultrasonic/left` | `Float32` | `ULTRASONICOS` |
| `/sensors/ultrasonic/center` | `Float32` | `ULTRASONICOS` |
| `/sensors/ultrasonic/right` | `Float32` | `ULTRASONICOS` |
| `/sensors/limit_switches` | `Int32MultiArray` | `FINALCARRERA` |
| `/motors/rpm` | `Float32MultiArray` | `MOTORES` |
| `/motors/ticks` | `Int32MultiArray` | `MOTORES` |
| `/motors/time_ms` | `Float32` | `MOTORES` |
| `/actuators/trimmer_state` | `Bool` | `TRIMMER` |
| `/actuators/brushless_state` | `Bool` | `BRUSHLESS` |
| `/actuators/buzzer_text` | `String` | `BUZZER` |
| `/actuators/display_text` | `String` | `DISPLAY` |
| `/imu/data_raw` | `Imu` | `IMU` |
| `/imu/angles` | `Float32MultiArray` | `IMU` |
| `/battery/voltage` | `Float32` | `BATERIA` |
| `/safety/estop` | `Bool` | `ESTADO` |
|

- Suscripciones principales:

| Topic | Tipo | Comando serial generado |
|---|---|---|
| `/esp32/cmd_motores` | `Float32MultiArray` | `CMD_MOTORES,left,right` |
| `/esp32/cmd_trimmer` | `Bool` | `CMD_TRIMMER,0/1` |
| `/esp32/cmd_blade` | `Bool` | `CMD_BLADE,0/1` |
| `/esp32/cmd_display` | `String` | `CMD_DISPLAY,text` |
| `/esp32/cmd_buzzer` | `String` | `CMD_BUZZER,text` |
| `/esp32/cmd_estop` | `Empty` | `CMD_ESTOP` |
| `/esp32/cmd_reset_estop` | `Empty` | `CMD_RESET_ESTOP` |
| `/esp32/cmd_reset_encoders` | `Empty` | `CMD_RESET_ENCODERS` |
| `/esp32/cmd_imu_calibrate` | `Empty` | `CMD_IMU_CALIBRATE` |
| `/esp32/cmd_reset_yaw` | `Empty` | `CMD_RESET_YAW` |
| `/esp32/cmd_status` | `Empty` | `CMD_STATUS` |
| `/esp32/cmd_ping` | `Empty` | `CMD_PING` |
| `/esp32/motors_enable` | `Bool` | `CMD_MOTORS_ENABLE,0/1` |
|

### Algunos topicos importantes

| Tópico | Función |
|---|---|
| `/sensors/ultrasonic` | Publica las distancias medidas por los sensores ultrasónicos. |
| `/sensors/limit_switches` | Publica el estado de los finales de carrera o bumpers. Sirve para detectar choques físicos. |
| `/motors/rpm` | Publica las RPM reales medidas en los motores. Permite verificar si las ruedas giran como se espera. |
| `/motors/ticks` | Publica los ticks de los encoders. Sirve para estimar movimiento y comparar el avance de cada rueda. |
| `/imu/angles` | Publica roll, pitch, yaw y estado de calibración de la IMU. Se usa para conocer y corregir la orientación del robot. |
| `/battery/voltage` | Publica el voltaje de la batería. |
| `/esp32/cmd_motores` | Recibe las RPM deseadas para la rueda izquierda y derecha. Es el tópico principal para mover el robot. |
| `/esp32/cmd_trimmer` | Envía la orden para encender o apagar el motor trimmer. |
| `/esp32/cmd_blade` | Envía la orden para encender o apagar la cuchilla |
| `/navigation_app/mode` | Recibe desde la app el modo de navegación seleccionado, como manual, random, parallel_lines o perimetral. |
| `/start` | Recibe desde la app la orden de iniciar o detener el movimiento autónomo. |

En resumen, los tópicos de sensores transportan información desde el ESP32 hacia ROS2, mientras que los tópicos de comando permiten que ROS2 envíe órdenes al ESP32 para accionar motores y actuadores. Esta separación permite que la Raspberry Pi se encargue de la lógica de alto nivel, mientras que el ESP32 ejecuta el control directo del hardware.

---

#### `pkg_navegacion`

| Nodo | Función resumida |
|---|---|
| `perimetral_node.py` | Controla el modo perimetral siguiendo una pared derecha con sensores ultrasónicos, detectando curvas de 90°, obstáculos frontales y pérdida de pared. |
| `random_node.py` | Controla el modo aleatorio, avanzando, retrocediendo o girando según obstáculos, bumpers, IMU y opcionalmente campo potencial. |
| `parallel_lines_node.py` | Controla el modo de líneas paralelas, avanzando recto con corrección por IMU, realizando giros programados y reaccionando ante obstáculos o bumpers. |
| `fence_node.py` | Recibe el polígono del cerco virtual y la posición GPS del robot para validar si el cerco es correcto y publicar si el robot está dentro o fuera. |
| `robot_gps_track_node.py` | Registra periódicamente las coordenadas GPS del robot mientras la navegación está activa para mostrar el recorrido realizado en el mapa. |

##### `fence_node`

- Tópicos que publica

| Topic | Tipo | Funcion |
|---|---|---|
| `/fencenode/valid` | `Bool` | Indica si el cerco virtual recibido es válido |
| `/fencenode/robot_dentro` | `Int32` | Indica si el robot está dentro del cerco: `1` dentro, `0` fuera |
| `/fencenode/status_text` | `String` | Estado textual del cerco y posición del robot |

- Tópicos a los que se suscribe

| Topic | Tipo | Funcion |
|---|---|---|
| `/fence/polygon` | `Polygon` | Polígono del cerco virtual enviado desde la app |
| `/gps/fix` | `NavSatFix` | Posición GPS actual del robot |

##### `parallel_lines_node`

Este nodo contiene toda la logica correspondiente al modo de navegacion de lineas paralelas. Consiste en realizar el corte siguiendo un patron de lineas paralelas, basando su movimiento en la lectura de los sensores ultrasonicos y bumpers, asi tambien con el IMU para mantener la direccion y manejar los angulos de giro.

El nodo funciona mediante una maquina de estados, el cual en su loop, primero lee las entradas (sensores y datos de la app, si se presiono el botón de inicio, si el cerco es valido, etc), decide la activación del disco, toma la decision de en que estado sera el nuevo según las entradas, ejecuta las acciones dependiendo del estado, y publica en la app el estado en que se encuentra. Luego se encuentran las funciones auxiliares del nodo.


- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/parallel_lines/status` | `String` | Publica el estado actual del modo de líneas paralelas. |
| `/parallel_lines/debug` | `String` | Publica mensajes de depuración y transiciones de estado. |
| `/esp32/cmd_motores` | `Float32MultiArray` | Publica las RPM calculadas para mover las ruedas. |
| `/esp32/cmd_blade` | `Bool` | Publica la orden para encender o apagar la cuchilla. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/navigation_app/mode` | `String` | Recibe el modo seleccionado desde la app. |
| `/fencenode/valid` | `Bool` | Recibe si el cerco virtual es válido. |
| `/start` | `Bool` | Recibe la orden de inicio o detención. |
| `/fencenode/robot_dentro` | `Int32` | Recibe si el robot está dentro del cerco virtual. |
| `/imu/angles` | `Float32MultiArray` | Recibe la orientación del robot para corregir avance y giros. |
| `/sensors/ultrasonic` | `Float32MultiArray` | Recibe distancias para detectar obstáculos. |
| `/sensors/limit_switches` | `Int32MultiArray` | Recibe el estado de los bumpers o finales de carrera. |

##### `perimetral_node`

Este nodo controla el modo de navegación perimetral del robot, haciendo que avance siguiendo una pared del lado derecho (lado donde se encuentra el trimmer) mediante sensores ultrasónicos. Para esto utiliza las distancias medidas por el sensor frontal, derecha delantera y derecha trasera, calcula si el robot está cerca, lejos o torcido respecto a la pared, y corrige las RPM de cada rueda publicando comandos en `/esp32/cmd_motores`. Además, detecta obstáculos frontales, curvas de 90° hacia la derecha y pérdida de pared, cambiando de estado según cada situación. También publica información de estado y debug, y activa el trimmer mientras el modo perimetral está en funcionamiento.

- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/perimetral/status` | `String` | Publica el estado actual del modo perimetral. |
| `/perimetral/debug` | `String` | Publica mensajes de depuración y razones de cambio de estado. |
| `/esp32/cmd_motores` | `Float32MultiArray` | Publica las RPM calculadas para seguir la pared derecha. |
| `/esp32/cmd_trimmer` | `Bool` | Publica la orden para encender o apagar el trimmer. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/navigation_app/mode` | `String` | Recibe el modo seleccionado desde la app. |
| `/start` | `Bool` | Recibe la orden de inicio o detención. |
| `/sensors/ultrasonic` | `Float32MultiArray` | Recibe las distancias usadas para seguir pared y detectar obstáculos. |

##### `random_node`

Este nodo contiene toda la lógica correspondiente al modo de navegación aleatorio. Consiste en realizar el corte de forma aleatoria basando sus movimientos en la lectura de los sensores ultrasónicos y bumpers, así también con el IMU, para mantener la dirección y manejar los ángulos de giro. Tambien es posible activar el campo potencial, el cual calcula un vector resultante de fuerza de la suma vectorial de los vectores de atracción y repulsión. Estos a su vez se calculan según la distancia con el robot, el umbral a partir del cual influyen, el ángulo con respecto al robot y la constante de proporcionalidad para cada uno. De este vector resultante, extraemos el angulo con el que el robot debe inclinarse para evitar los obstáculos.
El nodo funciona mediante una maquina de estados, el cual en su loop, primero lee las entradas (sensores y datos de la app, como si el campo esta activo, si se presiono el botón de inicio, si el cerco es valido, etc), decide la activación del disco, toma la decision de en que estado sera el nuevo según las entradas, ejecuta las acciones dependiendo del estado, y publica en la app el estado en que se encuentra. Luego se encuentran las funciones auxiliares del nodo.
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/random/status` | `String` | Publica el estado actual del modo perimetral. |
| `/random/debug` | `String` | Publica mensajes de depuración y razones de cambio de estado. |
| `/esp32/cmd_motores` | `Float32MultiArray` | Publica las RPM calculadas para seguir la pared derecha. |
| `/navegacion/corte` | `Bool` | Publica la orden para encender o apagar el trimmer. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/navigation_app/mode` | `String` | Recibe el modo seleccionado desde la app. |
| `/fencenode/valid` | `Bool` | Recibe si es valido el cerco enviado. |
| `/start` | `Bool` | Recibe la orden de inicio o detención. |
| `/fencenode/robotdentro` | `Bool` | Recibe si el robot se encuentra dentro del robot. |
| `/potential_field/enable` | `Bool` | Recibe si el campo potencial esta o no activado. |
| `/imu/angles` | `Float32MultiArray` | Recibe roll, pitch, yaw y si el IMU fue calibrado. |
| `/sensors/ultrasonic` | `Float32MultiArray` | Recibe las distancias de los ultrasónicos en forma de vector. |
| `/sensors/limit_switches` | `Int32MultiArray` | Recibe el estado de los bumpers en forma de vector. |

##### `robot_gps_track_node`

Este nodo se encarga de publicar la posicion del robot cuando un modo de navegacion ha sido iniciado, a fin de que la app pueda leer estas coordenadas y poder graficar la trayectoria recorrida por el robot.
Este nodo comienza a publicar la posicion que escucha desde el nodo de /gps/fix, de forma periodica, cuando el topic /start/ o /robot/gps_track_enable tienen como dato true.
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/robot/gps_track` | `Float64MultiArray` | Publica la lista de coordenadas GPS recorridas por el robot. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/gps/fix` | `NavSatFix` | Recibe la posición GPS actual del robot. |
| `/start` | `Bool` | Recibe la orden de inicio o detención de la navegación. |
| `/navigation_app/mode` | `String` | Recibe el modo de navegación seleccionado desde la app. |
| `/robot/gps_track_enable` | `Bool` | Permite activar manualmente el registro del recorrido GPS. |

---

#### `pkg_sensores`

| Nodo | Función |
|---|---|
| `final_carrera_node.py` | Lee los seis finales de carrera directamente desde los GPIO de la Raspberry Pi y publica su estado en `/sensors/limit_switches`.
 |
| `gps_uart_node.py` | Lee los datos NMEA del GPS por UART, convierte latitud y longitud a coordenadas decimales y publica posición, velocidad y cantidad de satélites en tópicos ROS2. |


##### `final_carrera_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/sensors/limit_switches` | `Int32MultiArray` | Publica el estado de los seis finales de carrera conectados a la Raspberry Pi. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| Ninguno | - | Lee directamente los GPIO de la Raspberry Pi. |

##### `gps_uart_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/gps/fix` | `NavSatFix` | Publica la posición GPS del robot. |
| `/gps/speed` | `Float32` | Publica la velocidad calculada a partir del GPS. |
| `/gps/satellites` | `Int32` | Publica la cantidad de satélites detectados. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| Ninguno | - | Lee directamente los datos NMEA del GPS por UART. |

---


#### `pkg_actuadores`

| Nodo | Función |
|---|---|
| `blade_node.py` | Controla el encendido y apagado de la cuchilla brushless según la solicitud de navegación, el modo manual y la detección de pasto por visión artificial.
 |
| `buzzer_node.py` | Reproduce avisos sonoros en el ESP32 cuando cambia el modo de navegación, inicia o detiene el robot, se activa el estop o la batería está baja.|
| `motor_driver_node.py` | Convierte los comandos de velocidad `/cmd_vel` en RPM para cada rueda y los publica en `/esp32/cmd_motores` para mover el robot. |

##### `blade_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/esp32/cmd_blade` | `Bool` | Publica la orden final para encender o apagar el brushless. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/vision/enabled` | `Bool` | Recibe si la visión artificial está activada desde la app. |
| `/grass_state` | `String` | Recibe si se detecta pasto o no. |
| `/navegacion/corte` | `Bool` | Recibe la solicitud de corte desde el modo de navegación. |
| `/vision/manual_blade_enable` | `Bool` | Recibe la activación manual de la cuchilla desde la app. |

##### `buzzer_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/esp32/cmd_buzzer` | `String` | Publica el comando de sonido que debe reproducir el buzzer. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/navigation_app/mode` | `String` | Recibe el modo seleccionado desde la app. |
| `/start` | `Bool` | Recibe si el robot inició o se detuvo. |
| `/safety/estop` | `Bool` | Recibe si la parada de emergencia está activa. |
| `/battery/voltage` | `Float32` | Recibe el voltaje de batería para activar alerta de batería baja. |

##### `motor_driver_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/esp32/cmd_motores` | `Float32MultiArray` | Publica las RPM izquierda y derecha calculadas desde `/cmd_vel`. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/cmd_vel` | `Twist` | Recibe velocidad lineal y angular para mover el robot. |

---

#### `pkg_vision`

| Nodo | Función |
|---|---|
| `camera_bridge_node.py` |  Toma el video desde el stream de la cámara, lo convierte a mensajes ROS2 tipo `Image` y lo publica en `/camera/image_raw`. |
| `grass_detector_node.py` | Recibe la imagen de la cámara, analiza una región inferior de la imagen con un modelo `.tflite` y publica si detecta `pasto` o `no pasto` en `/grass_state`. |
| `object_detection_node.py` | Recibe la imagen de la cámara, analiza una región frontal con un modelo `.tflite` de obstáculos y publica si detecta o no un obstáculo en `/vision_obstacle_state`. |

##### `camera_bridge_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/camera/image_raw` | `Image` | Publica la imagen capturada desde el stream de la cámara. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| Ninguno | - | Lee directamente el stream de video `tcp://127.0.0.1:8888`. |
##### `grass_detector_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/grass_state` | `String` | Publica el resultado de clasificación: `pasto`, `no pasto` o `none`. |
| `/grass/debug_image` | `Image` | Publica una imagen de debug con la región analizada marcada. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/camera/image_raw` | `Image` | Recibe la imagen de la cámara para clasificar presencia de pasto. |
##### `object_detection_node`
- Tópicos que publica

| Topic | Tipo | Función |
|---|---|---|
| `/vision_obstacle_state` | `String` | Publica el resultado de detección de obstáculos. |
| `/obstacle/debug_image` | `Image` | Publica una imagen de debug con las detecciones marcadas. |

- Tópicos a los que se suscribe

| Topic | Tipo | Función |
|---|---|---|
| `/camera/image_raw` | `Image` | Recibe la imagen de la cámara para detectar obstáculos. |

---

#### `pkg_bringup`

| Nodo/launch | Función |
|---|---|
| `robot_bringup.launch.py` | Arranca automáticamente el sistema base del robot, conectando el ESP32, sensores, actuadores, GPS, cámara, modelos de visión artificial, servidor web de video, cerco virtual, registro GPS y comunicación con la app mediante `rosbridge`. |

El archivo launch no publica ni se suscribe directamente a tópicos. Su función es iniciar varios nodos del sistema al mismo tiempo: bridge del ESP32, GPS, finales de carrera, cámara, visión artificial, buzzer, cuchilla, cerco virtual, track GPS, servidor de video y rosbridge para la app.

## 11. **APLICACIÓN**

### 11.1 Rol de la app

La aplicacion se desarrollo utilizando Flutter, un conjunto de herramientas (SDK) y framework de código abierto creado por Google, debido a que permite a los desarrolladores crear aplicaciones nativas de alta calidad para iOS, Android, Web, Windows, macOS y Linux, utilizando una única base de código.

La aplicacion le permite al usuario controlar y monitorear el robot desde un teléfono. Se tiene 5 pestañas dentro de la app: Inicio, Operacion, Mapa, Sensores y Configuracion.

Dentro de la pestaña de inicio podemos ver si esta conectado al robot en la barra superior, también visualizar la camara según la URL para poder streamear los tópicos, tanto de detección de pasto o de obstáculos, como un toggle el cual permite a la visión enviar ordenes de corte y no corte para el disco con cuchillas. También se ve el estado de la bateria, cuanto tiempo puede cortar aun el robot y el ultimo evento que esta realizando e incluso haciendo click sobre ese widget se pueden visualizar los eventos anteriores. Finalmente tenemos un widget de Estado + Debug, util para la corrección de código para los modos de navegación.

En la pestaña de mapa, vemos visualmente, como su nombre dice, el mapa, proporcionado por'OpenStreetMap', el cual es un proyecto colaborativo y de código abierto para crear un mapa editable y gratuito del mundo. Miles de voluntarios utilizan dispositivos GPS, imágenes satelitales y su conocimiento local para trazar calles, edificios y lugares de interés en todo el planeta.
Dentro de este podemos definir el cerco virtual, que seria el area en el cual queremos que la cuchilla este encendida y asi tambien podemos ver como se mueve en tiempo real el robot al realizar la tarea.

<div align="center">
  <table>
    <tr>
      <td style="text-align:center;">
        <img src="https://drive.google.com/uc?export=view&id=1JEJZYZAuJOsL2fPrAP7svfip-OxMIPgE" width="200"/>
        <p><b>Fig. 16.</b> Inicio</p>
      </td>
      <td style="text-align:center;">
        <img src="https://drive.google.com/uc?export=view&id=1Z_u5hy52ldO8r387rNxEKd88kHxxe1DI" width="200"/>
        <p><b>Fig. 17.</b> Mapa</p>
      </td>
    </tr>
  </table>
</div>
En cuanto a la pestaña de operación, en esta encontramos los distintos modos de funcionamiento mediante botones, dependiendo de cual se elija se habilitan mas opciones o no. El modo manual dispone widgets adicionales que se despliegan una vez que se selecciona el mismo, los cuales serian: la velocidad, el joystick, y botones para los actuadores. Para el joystick básicamente se calcula la posición del touch respecto al centro del mismo, mediante el radio del widget, eso para sacar la posición xy (adelante y giro) del touch, esto se normaliza de -1 a 1, se saca el porcentaje de la velocidad que decidió el usuario, y se multiplica este por el resultado del calculo de RPMs de la rueda izquierda y derecha, el cual se hace según el modelo diferencial. El modo aleatorio permite encender el campo potencial, el cual la base teórica es muy similar a la del campo eléctrico, con la atracción y repulsion de cargas eléctricas. El video del cual nos basamos para la aplicación matemática se encuentra en la biografía. Finalmente, los modos cuentan con un botón de Inicio y Detener, para el corte. Al iniciar un corte, si el usuario no se encuentra en la pestaña de Operación, el botón de Detener, se despliega en la barra superior de la app.

Por ultimo tenemos las pestañas de sensores y configuracion, en la de sensores visualizamos el estado de los mismos, ultrasónicos, que valores de medición arrojan, bumpers, GPS, nivel de batería, etc. Tambien podemos Calibrar y resetear el yaw del IMU. En configuración basicamente definimos la IP y asi tambien el puerto ded comunicacion con el robot. Disponemos de unas IPs guardadas para una conexión mas fácil a la hora de cambiar de redes.

<div align="center">
  <table>
    <tr>
      <td style="text-align:center;">
        <img src="https://drive.google.com/uc?export=view&id=1-xA1OMrltKLnCDY6HVeE5NbRu0P8XCB2" width="200"/>
        <p><b>Fig. 18.</b> Sensores</p>
      </td>
      <td style="text-align:center;">
        <img src="https://drive.google.com/uc?export=view&id=1udP-DgRRmMDDjlLDpORPP-w6PO9qxHR6" width="200"/>
        <p><b>Fig. 19.</b> Operación</p>
      </td>
    </tr>
  </table>
</div>

<div align="center"> <img src="https://drive.google.com/uc?export=view&id=1IgXBnHIOc8jAY1Gugv8YmrbDK489ga4j" width="200"/>
<p><b>Fig. 20.</b> Configuracion.</p> </div>

\
Para explicar un poco mas la programación, cada archivo suele organizarse de forma parecida. Al inicio se importan las librerías necesarias, por ejemplo material.dart, archivos de configuración, servicios o widgets propios. Después normalmente aparece la clase principal del widget, que representa una pantalla o un componente de la interfaz.
Cuando la pantalla necesita manejar datos que cambian, se usa un StatefulWidget, básicamente, una clase ya definida en el framework que indica cuando una pantalla cambia y tiene lógica detrás. En ese caso hay dos clases: una clase pública que define qué datos recibe el widget, y otra clase de estado que contiene la lógica interna. En esa clase de estado se suelen definir variables, controladores, funciones de inicialización como initState(), funciones de limpieza como dispose(), funciones auxiliares y finalmente el método build().
El método build() es el encargado de construir visualmente la pantalla. Cada vez que cambia un estado importante y se llama a setState(), Flutter vuelve a ejecutar build() y actualiza la interfaz. Por eso la lógica y la interfaz están relacionadas: las funciones modifican variables de estado, y luego build() usa esas variables para decidir qué widgets mostrar, qué textos poner, qué colores usar o qué botones habilitar.
En esta app, el flujo principal comienza en main.dart, que inicializa la aplicación Flutter y carga la primera pantalla. Luego se muestra LoadingScreen, que se encarga de preparar la conexión con el robot. Esta pantalla intenta conectarse con ROS mediante WebSocket. Si la conexión se logra, LoadingScreen deja de mostrarse y se reemplaza por HomeScreen, que es la pantalla principal de la app.
HomeScreen funciona como el centro de coordinación. No contiene toda la interfaz directamente, sino que organiza las pestañas principales y les pasa los datos y funciones que necesitan. Por ejemplo, le pasa a la pestaña de sensores los valores actuales del robot, a la pestaña de mapa los puntos del cerco y la posición GPS, y a la pestaña de operación las funciones para iniciar, detener o cambiar de modo.
Para que el código sea más ordenado, las ventanas principales se separaron en archivos propios, como home_tab.dart, map_tab.dart, operation_tab.dart, sensors_tab.dart y config_tab.dart. Cada uno de esos archivos define una parte grande de la interfaz. Así HomeScreen no queda cargado con todo el diseño visual de la app, sino que actúa más como controlador general.
Además, algunos widgets grandes o reutilizables también se separaron en archivos independientes. Por ejemplo, operation_tab.dart usa el widget Control Manual, que está en manual_control.dart, porque el joystick y los controles manuales tienen suficiente lógica y diseño como para vivir en su propio archivo. Algo parecido pasa con el mapa: map_tab.dart organiza la pestaña, pero el widget visual del mapa está separado en fence_map.dart.

\
Ahora, los archivos mas importantes en cuanto a la comunicación con ROS:
robot_connection.dart es el archivo encargado de la comunicación entre la app Flutter y ROS2. La app no se conecta directamente como un nodo ROS, sino que abre un WebSocket hacia rosbridge_server, usando una dirección del tipo ws://IP_RASPBERRY:9090. Rosbridge recibe los mensajes JSON desde Flutter y los traduce a topics reales de ROS2.
Dentro de este archivo se manejan cuatro tareas principales:
1. Conectarse al WebSocket.
2. Suscribirse a los topics de ROS.
3. Parsear los mensajes recibidos.
4. Publicar comandos hacia ROS.

\
Cuando la conexión se establece, la app envía mensajes de suscripción. Por ejemplo, se suscribe a topics como /gps/fix, /imu/angles, /sensors/ultrasonic, /safety/estop, entre otros. Cada vez que ROS publica algo en esos topics, rosbridge lo manda a la app como un JSON.
Luego entra la lógica de parseo. El archivo recibe el mensaje, lee de qué topic viene y entra en el case correspondiente. Por ejemplo, si llega un mensaje desde /gps/fix, se extraen la latitud, longitud y estado del fix. Si llega desde /imu/angles, se leen roll, pitch, yaw y calibración. Si llega desde /sensors/ultrasonic, se toman las distancias de los sensores. En cada caso, el dato recibido se transforma al formato que necesita la app.
Después de parsear un mensaje, robot_connection.dart actualiza el estado llamando a funciones de robot_state.dart, como updateGps, updateBattery, updateImuAngles, updateUltrasonics, etc.
Además, robot_connection.dart también tiene funciones send..., que son las encargadas de publicar comandos desde la app hacia ROS. Por ejemplo:

*   sendMotorRpm() publica en /esp32/cmd_motores
*   sendTrimmerCommand() publica en /esp32/cmd_trimmer
*   sendBladeCommand() publica en /esp32/cmd_blade
*   sendNavigationMode() publica en /navigation_app/mode
*   sendEmergencyStop() publica en /esp32/cmd_estop

Estas funciones arman un mensaje JSON con el topic y los datos, y lo mandan por WebSocket a rosbridge.
robot_state.dart, en cambio, no se comunica con ROS. Su trabajo es guardar el estado actual del robot dentro de la app. Ahí se almacenan valores como batería, GPS, IMU, ultrasonidos, bumpers, estado de emergencia, actuadores, eventos y datos de navegación.
La relación entre ambos archivos es esta:

\
ROS2 → rosbridge → robot_connection.dart → robot_state.dart → interfaz Flutter

\
Cuando llega un dato nuevo desde ROS, robot_connection.dart lo interpreta y llama a un método update... de robot_state.dart. Ese método guarda el nuevo valor y ejecuta notifyListeners(). Con eso Flutter sabe que el estado cambió y actualiza las partes de la pantalla que dependen de esos datos.

### 11.2 Archivos de la app

| Archivo | Responsabilidad |
|---|---|
| `main.dart` | Entrada de la app y configuración inicial de `MaterialApp`. |
| `config/app_config.dart` | IP, puertos, modo simulación, perfiles guardados, URL WebSocket y URLs de cámara. |
| `services/robot_connection.dart` | Conexión WebSocket con rosbridge, suscripciones a topics y publicación de comandos hacia ROS2. |
| `services/robot_state.dart` | Estado del robot: sensores, GPS, IMU, actuadores, eventos y estado de navegación. |
| `screens/loading_screen.dart` | Pantalla inicial de conexión/carga. |
| `screens/home_screen.dart` | Contenedor principal: coordina pestañas, conexión, modos, eventos, mapa, cámara y comandos globales. |
| `screens/tabs/home_tab.dart` | Pantalla de inicio: estado general, cámara, visión, debug y eventos. |
| `screens/tabs/operation_tab.dart` | Pantalla de operación: selección de modo, inicio/detención y control manual. |
| `screens/tabs/map_tab.dart` | Mapa, cerco virtual, trayectoria del robot y estado del cerco. |
| `screens/tabs/sensors_tab.dart` | Visualización de sensores: batería, GPS, IMU, ultrasonidos y bumpers. |
| `screens/tabs/config_tab.dart` | Configuración de IP, puerto, modo debug, duración y perfiles de conexión. |
| `widgets/camera_status.dart` | Visualización de cámara MJPEG, cambio de stream y fullscreen. |
| `widgets/manual_control.dart` | Joystick manual, trimmer y brushless. |
| `widgets/fence_map.dart` | Mapa con cerco, robot y recorridos dibujados. |

### 11.3 Tópicos a los cuales la app debe suscribirse

| Dato | Topic ROS2 actual | Tipo | Comentario |
|---|---|---|---|
| Voltaje batería | `/battery/voltage` | `std_msgs/Float32` | La app calcula porcentaje desde el voltaje. |
| GPS | `/gps/fix` | `sensor_msgs/NavSatFix` | Usa latitud, longitud y `status.status` para saber si hay fix. |
| Satélites GPS | `/gps/satellites` | `std_msgs/Int32` | Se muestra como cantidad de satélites disponibles. |
| Trayectoria GPS robot | `/robot/gps_track` | `std_msgs/Float64MultiArray` | Lista `[lat1, lon1, lat2, lon2, ...]` para dibujar recorrido. |
| IMU | `/imu/angles` | `std_msgs/Float32MultiArray` | Orden esperado: `[roll, pitch, yaw, calibrated]`. |
| Ultrasonidos | `/sensors/ultrasonic` | `std_msgs/Float32MultiArray` | Orden actual: izquierda, frente, derecha, diagonal derecha, diagonal izquierda, trasero derecha. |
| Bumpers / finales de carrera | `/sensors/limit_switches` | `std_msgs/Int32MultiArray` | Orden actual: left, right, back, center left, center right, down. |
| Emergencia | `/safety/estop` | `std_msgs/Bool` | Indica si el robot está en parada de emergencia. |
| Estado trimmer | `/actuators/trimmer_state` | `std_msgs/Bool` | Estado real informado desde ESP32/ROS. |
| Estado brushless | `/actuators/brushless_state` | `std_msgs/Bool` | Estado real informado desde ESP32/ROS. |
| Estado cerco | `/fencenode/status_text` | `std_msgs/String` | Texto de estado del nodo de cerco. |
| Robot dentro del cerco | `/fencenode/robot_dentro` | `std_msgs/Int32` | `1` dentro, `0` fuera, otro valor indefinido. |
| Estado Random | `/random/status` | `std_msgs/String` | Estado del modo random. |
| Debug Random | `/random/debug` | `std_msgs/String` | Eventos/debug del modo random. |
| Estado Líneas Paralelas | `/parallel_lines/status` | `std_msgs/String` | Estado del modo líneas paralelas. |
| Debug Líneas Paralelas | `/parallel_lines/debug` | `std_msgs/String` | Eventos/debug del modo líneas paralelas. |
| Estado Perimetral | `/perimetral/status` | `std_msgs/String` | Estado del modo perimetral. |
| Debug Perimetral | `/perimetral/debug` | `std_msgs/String` | Eventos/debug del modo perimetral. |

### 11.4 Comandos desde la app

| Acción en app | Topic ROS2 actual | Tipo |
|---|---|---|
| Joystick / movimiento manual | `/esp32/cmd_motores` | `std_msgs/Float32MultiArray` |
| Iniciar / detener navegación | `/start` | `std_msgs/Bool` |
| Habilitar track GPS manual | `/robot/gps_track_enable` | `std_msgs/Bool` |
| Activar trimmer | `/esp32/cmd_trimmer` | `std_msgs/Bool` |
| Activar brushless | `/esp32/cmd_blade` | `std_msgs/Bool` |
| Calibrar IMU | `/esp32/cmd_imu_calibrate` | `std_msgs/Empty` |
| Reset yaw IMU | `/esp32/cmd_imu_reset_yaw` | `std_msgs/Empty` |
| Modo de navegación | `/navigation_app/mode` | `std_msgs/String` |
| Texto display ESP32 | `/esp32/cmd_display` | `std_msgs/String` |
| Campo potencial | `/potential_field/enable` | `std_msgs/Bool` |
| Activar visión automática | `/app/vision_enable` | `std_msgs/Int32` |
| Permiso manual brushless con visión | `/vision/manual_blade_enable` | `std_msgs/Bool` |
| Enviar cerco | `/fence/polygon` | `geometry_msgs/Polygon` |

### 11.5 Enlace de video y drive

https://www.youtube.com/watch?v=RoeXswWy8GY
\
https://drive.google.com/drive/folders/1KKlexMRB7UoI95Z5T-rhAGOylLbZ30FX?hl=pt-BR

## 12. **FLUJO COMPLETO DEL SISTEMA**

El flujo final de funcionamiento puede resumirse así:

```text
[Usuario]
   ↓
[App Flutter]
   ↓ WebSocket
[rosbridge_server]
   ↓ ROS2 topics
[Raspberry Pi / ROS2]
   ↓ /esp32/cmd_*
[pkg_esp32_bridge]
   ↓ Serial USB 115200
[ESP32 / Firmware]
   ↓ GPIO, PWM, I2C, UART
[Actuadores y sensores]
```

Y en sentido inverso:

```text
[Sensores / Encoders / GPS / IMU]
   ↓
[ESP32]
   ↓ Serial telemetry
[esp32_bridge_node]
   ↓ ROS2 topics
[Nodos ROS2 + App Flutter]
```

Este diseño permite que cada parte tenga una responsabilidad clara y evita que varios programas intenten abrir el puerto serial del ESP32 al mismo tiempo.

## 13. **EVIDENCIAS DE DOCUMENTACION, VIDEOS Y CODIGOS**

| Evidencia | Descripción | Enlace |
|---|---|---|
| Código firmware ESP32 | Proyecto PlatformIO completo | https://drive.google.com/drive/folders/1cVjHoy63RicNz62-Doxq0xT7jPzCsn6v?usp=drive_link|
| Código ROS2 Raspberry | Workspace `brain_captainp` con paquetes | https://drive.google.com/drive/folders/1wHy5k62tUHLCbyhy4WjuMzZdt2zhb8Vj?usp=drive_link|
| Código app Flutter | Proyecto Flutter Captain Pingui |https://drive.google.com/drive/folders/1KKlexMRB7UoI95Z5T-rhAGOylLbZ30FX?usp=drive_link|`
| Modelo entrenado `.pt` | Pesos YOLOv8 finales | https://drive.google.com/file/d/1jDoo6j-QCAO0HuFkw3iRjX85aN4FDFWT/view?usp=drive_link |
| Modelo `.tflite` | Exportación para Raspberry Pi | https://drive.google.com/file/d/1dPj5gRq5EoJ32QMg07fkbi0beg9-aujB/view?usp=drive_link |
| Foto demo montaje | Robot físico y componentes | https://drive.google.com/file/d/1f3-8nPf5FJYVnYmP33Sb3NwJdA-DS76u/view?usp=drive_link |
| Video Modo Random | Movimiento aleatorio del Robot. | https://drive.google.com/file/d/1Qxcm8DSAGsiP_i89CqkBff9DUgsp5CQH/view?usp=drive_link |
| Video Modo Manual | Control manual del Robot desde la APP.| https://drive.google.com/drive/folders/1WhtY80wwucgRMdjZwaovr3GDlLdLpdLA?usp=drive_link|
| Video Modo Lineas Paralelas | Movimiento en un patron controlado de lineas paralelas. | https://drive.google.com/drive/folders/1767sUef9U2eMUDaI1O1NHOanCQ5eOn6o?usp=drive_link|
| Video Modo Perimetral | Movimiento siguiendo el perimetro de un area. | https://drive.google.com/drive/folders/1sWoTwd6CMJIH-MezreUf3fZtVGLe3fk4?usp=drive_link|


In [ ]:
# SONIDOS Y DISPLAY
from IPython.display import HTML

video_id = "1kxJLPUwpNxy-J2Lfk8aobl0kY3I1H0KN"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

In [ ]:
# MODO RANDOM CON CAMPO POTENCIAL
from IPython.display import HTML

video_id = "1CIYdB2paJYf-pesrik5pEzQKQKu1gcaV"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

In [ ]:
# MODO RANDOM SIN CAMPO POTENCIAL
from IPython.display import HTML

video_id = "1P9EalkKsTPmv7eZdP5gBX9T9HN3ndl3c"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

In [ ]:
# MODO LINEAS PARALELAS
from IPython.display import HTML

video_id = "1WyrYRFOk35NiWQcVkizzHhval3mNqW75"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

In [ ]:
# MODO PERIMETRAL
from IPython.display import HTML

video_id = "1LwElRl8Za7E_1DKNzq6uzCkH35jgJlfF"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

In [ ]:
# MODO MANUAL
from IPython.display import HTML

video_id = "1QqOrvNJX5azm2EsnbVLRee5-lY5a7qkN"  # reemplaza con tu ID
HTML(f"""
<iframe src="https://drive.google.com/file/d/{video_id}/preview" width="640" height="480" allow="autoplay"></iframe>
""")

## 14. **PROBLEMAS Y POSIBLES MEJORAS**

***DISEÑO - MECANICO***
*   Distribucion de componentes: A la hora de diseñar el cuerpo del robot no se tuvo en cuenta la distribucion de los componentes, esto significa, la distribucion de peso. Por lo cual, al estar la mayoria de los componentes en la zona frontal este perdia traccion para moverse.\
SOLUCION: le agregamos unos dientes a la rueda para ayudar con la traccion y fuerza de empuje de las ruedas con respecto a la superficie.
*  Colocacion de las ruedas: Los motores DC Gusano, los cuales tienen como fin la locomoción diferencial del robot, pasaron por varios propotipos para su soporte. Finalmente, se llego a la conclusion que el mejor seria uno que lo coloque de forma vertical y "de cabeza". El problema con este soporte fue, que a la hora de su diseño, hubo una zona la cual no sujetaba bien al motor, como consecuencia, el motor bailaba dentro del soporte, lo cual no permitia que las ruedas queden fijas en el robot.
Otro problema fue que no se consideró en el diseño base y pared, como atornillar los tornillos prisioneros que irian a la unión acople + eje motor
\
SOLUCION: La forma mas rapida de evitar este problema sin aumentar los gastos, fue usar pedazos de carton para mantener firme el motor y alinear las ruedas.  Asi tambien se realizo agujeros en la base para facilitar el ajuste del tornillo prisionero de acople. Lo ideal hubiese sido rediseñar el modo de acople.
*   Vibracion de los componentes: El motor de corte (BLDC) generaba muchas vibraciones que se extendia a lo largo de todo el robot, algo que se supuso de antemano, generando ruido en diversos componentes electricos y afectando asi el funcionamiento del mismo.
\
SOLUCION: Se logró reducir la vibracion colocando pedazos de goma de una camara de rueda de moto, dentro de lo que seria el sistema de disco: entre el disco y la base, entre los tornillos y el disco, etc.
*  Diseño de bumpers: el diseño contemplaba unas guias para que todo el esfuerzo no recaiga en los resortes pero esta guia no tenia un correcto diseño de tal manera que deslice correctamente y la posicion de las mismas no era la adecuada.
\
SOLUCION: diseño de unas guias mejor ubicadas (en los extremos del bumper) y con una forma mas adecuada para el deslizamiento y movimiento lineal de todo el bumper. En la practica solucionamos sacando las guias y pegando con epoxy, pero los resortes pasaron a recibir los esfuerzos totales.

***CONEXIONADO ELECTRICO - FIRMWARE***

*   Bateria principal: Durante el ensamblaje del robot accidentalmente se realizo un cortocircuito entre el positivo y negativo de la bateria. Esto ocasiono un daño en el BMS, con lo cual la bateria "reflejaba" la carga que se le conectaba.
\
SOLUCION: se opto por utilizar una bateria de 12V para los motores de las ruedas y la parte logica, y dos baterias en serie para el BLDC para obtener 24V. Lo ideal hubiese sido cambiar el BMS pero el mismo no se encuentra disponible para venta en el pais y asi tambien, por cuestion tiempo ya no fue una opcion viable.

*   Bateria nueva introducida: Debido al suceso ocurrido con la bateria principal, la anatomia del robot se vió reflejada. Tanto la bateria de 12V y la del 24V nuevas, no tenian la capacidad suficiente para lograr la autonomia completa de 1h, pero fueron soluciones que se encontraron de manera rapida, de tal forma a poder continuar con el proceso del robot.

*   Cantidad de pines insuficientes: durante el desarrollo del proyecto tuvimos la necesidad de agregar mas sensores ultrasonicos, lo cual significa utilizar mas pines del controlador lo cual no teniamos.
\
SOLUCION: algunas conexiones pasamos al raspberry, como el GPS y los finales de carrera.
*   Acumulacion de conexiones en un pin: debido a la insuficiencia de pines disponibles colocamos los triggers de los ulltrasonicos en un mismo pin lo cual genero problemas en su funcionamiento ya que se volvio inestable su lectura, ya que al tener todos el mismo triggers se lanzan las ondas al mismo tiempo y asi los ultrasonicos son mas susceptibles en recibir ondas ajenas.
\
SOLUCION: Utilizar modulos expansores I2C para todos los echos y triggers, lo cual no pudimos implementar porque habia que realizar la importacion del componente y no habria llegado a tiempo.

*   Ruido EMI BLDC: al encender el motor brushless, el bus I2C comenzaba a arrojar errores de timeout (Error 263) e interrumpía la comunicación con la IMU y la pantalla OLED. Como el sistema cuenta con dos baterías separadas (12V para lógica y 24V para el brushless), al principio compartían un punto de masa común necesario para que la señal de control del motor tuviera referencia eléctrica respecto al ESP32. Esta masa común, sumada al ruido electromagnético generado por el driver y los cables de potencia del brushless al encenderse, introducía interferencias tanto en la línea de alimentación como en las señales SDA y SCL del bus I2C.\
SOLUCION: Se aplicaron tres correcciones en conjunto. Primero, se aumentó la velocidad de trabajo del brushless, lo que redujo la vibración mecánica del motor y estabilizó su operación, disminuyendo parte del ruido asociado al arranque. Segundo, se separó físicamente la referencia de masa de la lógica respecto a las masas de potencia de ambas baterías, aplicando el principio de masa en estrella: cada batería y el driver llevan su retorno de corriente directamente al punto de unión común, sin que esa corriente de alta potencia pase por el mismo tramo de cable que sirve de referencia a la lógica. Tercero, se apantallaron los cables de alimentación y señal del bus I2C, alejándolos físicamente de los cables de potencia del brushless. Con estas tres medidas combinadas, los errores de I2C desaparecieron y la IMU y el OLED operan correctamente de forma simultánea con el motor en funcionamiento.

*   Lectura erronea de los Ultrasonicos:El sensor ultrasónico presentaba lecturas inestables con saltos bruscos entre mediciones consecutivas. Esto se debía principalmente a ecos fantasmas, señales reflejadas en superficies secundarias que el sensor interpretaba como obstáculos reales, generando distancias que no correspondían al entorno real del robot.
\
Solución: Se implementaron filtros por software en el firmware del ESP32. Se agregó validación de ecos para descartar lecturas que cayeran fuera de un rango coherente respecto a mediciones anteriores, y un filtro de mediana para suavizar la señal y eliminar los picos espurios. Con esto las lecturas se estabilizaron y dejaron de aparecer los saltos abruptos.

*   Se quemaron varios componentes electronicos, tales como sensores ultrasonicos y modulo GPS, los cuales fueron reemplazados en el proceso.


***VISION***  
* Durante las primeras pruebas, los modelos utilizados para el nodo de vision artificial eran ejecutados mediante las librerías asociadas a PyTorch/Ultralytics (.pt). Esto resultó poco adecuado para nuestra Raspberry Pi 4, debido al alto costo computacional que implicaba usar las librerías. El problema generado era que el flujo de video presentaba retardos de casi 20 segundos y la cámara se actualizaba lentamente. Para nuestro robot móvil el retardo que teniamos ya era crítico, ya que la detección tardía producía decisiones incorrectas o fuera de tiempo.
\
SOLUCION: La solución aplicada fue convertir los modelos entrenados al formato .tflite, el cual está más orientado a aplicaciones de hardware limitado. Con TensorFlow Lite/AI Edge LiteRT, pudimos ejecutar el modelo de manera más liviana, reduciendo la latencia y mejorando la velocidad de procesamiento de imágenes. A partir de esto, se desarrollaron nodos de visión que analizan zonas específicas de la imagen en lugar de procesar todo el cuadro de forma innecesaria.

***APLICACION***
* Mejora en la conexión por IP: Si fuese un producto comercial, seria ineficiente que el usuario deba conocer la IP de la Raspberry dentro de la red para conectarse al robot.
\
SOLUCION:
Incorporar descubrimiento automático del robot en la red local. En lugar de exigir que el usuario conozca la IP de la Raspberry, el robot podría anunciarse mediante mDNS con un nombre fijo, como captainpingui.local. Así, la app se conectaría a ws://captainpingui.local:9090 sin depender de una IP que cambia según la red. También podría agregarse un escaneo automático de la red local para detectar el puerto del robot y guardar la conexión encontrada.


## 16. **Referencias**

- `TPF_Entrega1.ipynb`: introducción, arquitectura inicial, selección de componentes y primeras pruebas.
- `TPF_Entrega2.ipynb`: diseño y montaje de la plataforma.
- `TPF_Entrega3.ipynb`: plataforma montada y primeras pruebas de control.
- `TPF_Entrega4.ipynb`: actuadores, cálculo de autonomía y batería.
- `Entrega6.ipynb`: avance de visión artificial y detección de elementos del entorno.
- `R2_TPF2026S1.ipynb`: entrenamiento, inferencia y exportación de modelos YOLOv8.
- `Firmware PINGUI R2 — Documentación Técnica.pdf`: documentación del firmware modular del ESP32.
- `captain_pingui_informe_tecnico_app_ros2_ampliado.pdf`: documentación técnica de la app Flutter y su comunicación con ROS2.
- `
Derivation of Potential Field Method for Robot Path Planning and Obstacle Avoidance Using Lidar - Aleksandar Haber PhD` - Video Explicativo de la teoria de Campo Potencial https://www.youtube.com/watch?v=FT5DQ-SGYL4